# 01. LLM token 비용 모델

목표: 일반 input, output, cache write, cache read를 분리해 비용과 절감률을 계산합니다. 아래 단가는 학습용 가상 값이며 실제 AWS 가격이 아닙니다.

In [ ]:
PRICES_PER_MILLION = {
    "input": 3.00,
    "output": 15.00,
    "cache_write": 3.75,
    "cache_read": 0.30,
}

def cost_usd(usage, prices=PRICES_PER_MILLION):
    return sum(usage.get(kind, 0) * price / 1_000_000 for kind, price in prices.items())

before = {"input": 17_000 * 1_000, "output": 500 * 1_000}
after = {
    "input": 2_000 * 1_000,
    "output": 500 * 1_000,
    "cache_write": 15_000 * 20,
    "cache_read": 15_000 * 980,
}
before_cost = cost_usd(before)
after_cost = cost_usd(after)
print(f"before=${before_cost:.2f}, after=${after_cost:.2f}")
print(f"saving={(1-after_cost/before_cost):.1%}")

## Request hit rate와 token hit rate

두 지표의 분모가 다르므로 같은 dashboard에서 구분해야 합니다.

In [ ]:
def cache_rates(hit_requests, eligible_requests, read_tokens, write_tokens):
    request_rate = hit_requests / eligible_requests if eligible_requests else 0
    token_rate = read_tokens / (read_tokens + write_tokens) if read_tokens + write_tokens else 0
    return request_rate, token_rate

request_rate, token_rate = cache_rates(980, 1_000, after["cache_read"], after["cache_write"])
print(f"request hit rate={request_rate:.1%}")
print(f"token hit rate={token_rate:.1%}")

## Endpoint별 우선순위

최적화 가능 token과 예상 할인 폭을 곱해 큰 endpoint부터 정렬합니다.

In [ ]:
endpoints = {
    "attribute-extraction": {"tokens": 92_000_000, "fixed_ratio": 0.88},
    "customer-support": {"tokens": 5_000_000, "fixed_ratio": 0.35},
    "group-evaluation": {"tokens": 3_000_000, "fixed_ratio": 0.60},
}
ranking = sorted(
    ((name, item["tokens"] * item["fixed_ratio"]) for name, item in endpoints.items()),
    key=lambda row: row[1],
    reverse=True,
)
for name, cacheable in ranking:
    print(f"{name:22s} estimated cacheable tokens={cacheable:,.0f}")